In [ ]:
INPUT_WB_NAME  = "2026 Inputs for Apps.xlsx"

####
INPUT_WS_NAME  = "BEST OF TRADING INPUTS"
INPUT_TBL_NAME = "BEST_OF_INPUTS"
####

In [ ]:
# --- system setup ---
import sys
import os
sys.path.append(os.path.abspath(".."))

# --- autoreload ---
%load_ext autoreload
%autoreload 2

In [ ]:
import asyncio
from collections import defaultdict
#from datetime import datetime
import numpy as np
import pandas as pd
from zoneinfo import ZoneInfo
from IPython.display import display, clear_output

In [ ]:
# --- builders ---
from fin_insts import make_single_leg_fin_insts, BestOf #, FutureSpread, Synthetic

In [ ]:
# --- IBKR ---
from ibkr.Class_IBKR_IB import IBKR_IB
# from ibkr.Class_IBKR_TWS import IBKR_TWS
ibkr = IBKR_IB()

In [ ]:
# --- feeds ---
from ws_feeds import WSFeedManager

In [ ]:
# --- output ---
from output.Output_Methods import create_output
from output.Class_xlWings import xlWings
xlw = xlWings()

In [ ]:
# --- utils ---
# from other.Graph_Theory import find_all_node_permutations, connect_nodes_with_edges

In [ ]:
# --- trading strategy ---
from strategies import Strategy, BestOfStrat   

In [ ]:
# CONSTANTS

DB_WB_NAME  = "2026 Crypto Products Database.xlsx"

OUTPUT_COLS = [
               'time',
    
               'my_prod_type',
               'my_fi_name',
               'my_pf_name',
    
               'numerator_currency',
               'denominator_currency',
                              
               'price_mkt_bid',
               'price_mkt_ask',
    
               'scalar_price_mkt_to_unit',
    
               'price_unit_bid',
               'price_unit_ask'
            ]

In [ ]:
async def standard_startup(xlw, INPUT_WB_NAME, INPUT_WS_NAME, INPUT_TBL_NAME):

    df = xlw.get_df(INPUT_WB_NAME, INPUT_WS_NAME, INPUT_TBL_NAME, table=True)
    input_dict = df.set_index('Keys')['Values'].to_dict()
    
    wb  = input_dict['input workbook name']
    ws  = input_dict['true/false sheet name']
    tbl = input_dict['true/false table name']
    true_false_df = xlw.get_df(wb, ws, tbl, table=True)
    
    if 'TRUE/FALSE' not in true_false_df.columns:
        true_false_df = true_false_df.set_index('Keys').T

    true_false_df = true_false_df[true_false_df['TRUE/FALSE'] == True]
     
    wb  = DB_WB_NAME
####    
    ws  = input_dict['crypto long name']
    tbl = input_dict['crypto abbrev'] + "_static_data_table"
####
    
    db_df = xlw.get_df(wb, ws, tbl, table=True)
    
    merged_df = true_false_df.merge(db_df,how='left',on=['my_fi_name', 'my_pf_name'])

    fin_inst_objs_list = make_single_leg_fin_insts(merged_df)

    return input_dict, fin_inst_objs_list



In [ ]:
async def main():

    input_dict, objs_list = await standard_startup(xlw, INPUT_WB_NAME, INPUT_WS_NAME, INPUT_TBL_NAME)

    ws_objs_list = [obj for obj in objs_list if obj.my_pf_name != 'IBKR']
    ws_feed      = WSFeedManager(ws_objs_list)

    await ws_feed.complete_fi_objects()   
        
    ibkr_objs_list     = [obj for obj in objs_list if obj.my_pf_name == 'IBKR']
    if ibkr_objs_list:
        await ibkr.connect()
        print("IBKR connected:", ibkr.ib.isConnected())
        
        await asyncio.gather(*(ibkr.create_simple_contract(obj) for obj in ibkr_objs_list))
        await asyncio.gather(*(ibkr.complete_obj(obj) for obj in ibkr_objs_list))

# rarely change anything above here

    # bestOf setup
    attr_list = [('cf_unit_hit_bid_net',  max), 
                 ('cf_unit_lift_ask_net', max)]
    
    bo_obj = BestOf("ETF", ibkr_objs_list, attr_list, mode='auto')       
       
    # strategy setup
    strat = BestOfStrat(bo_obj)
        
    for obj in ibkr_objs_list:
        obj.platform_obj = ibkr  # this is the object not the name / can't be done in strat 
        
    # strat.print_orders = False
    
    output_list = [
        bo_obj, 
        *ibkr_objs_list,
                ]
    
    # run all streams concurrently
    tasks = []
    tasks.append(asyncio.create_task(create_output(input_dict, output_list, OUTPUT_COLS)))
    if ibkr_objs_list:
        tasks.append(asyncio.create_task(ibkr.start_streams(ibkr_objs_list)))
    
    await asyncio.sleep(15)

    await strat.done_event.wait()
    
    # then cancel everything else
    for task in tasks:
        task.cancel()
    
    # optional: wait for clean cancellation
    await asyncio.gather(*tasks, return_exceptions=True)
    
    # disconnect IBKR
    ibkr.ib.disconnect()
    
    print("Program finished cleanly.")
    #'''

In [ ]:
await main()